# E16 — Event composition as an acquired semantic unit
[Immutable pre-run protocol](https://github.com/shekhawatxyz/fastai-2022/blob/7c49a98267e74185eedbbfab22054cdb83e4b1a3/experiments/recursive_world/E16_PROTOCOL.md)

```
 observed A --> uncertain operation A --+
 observed B --> uncertain operation B --+--> C = unknown word(A,B)
                                             |       ^
                                             v       |
                                  D = unknown word(A,B,C)
                                             |
                                  observe D; revise C
```

Learn a joint prior over relationships from source worlds, not the source worlds' action identities. Every target action has an independent alternative explanation. The hypothesis language is still enumerated, not invented. This is conventional exact Bayesian inference; the experiment tests the value of acquired interdependencies, not an algorithmic novelty.

Text: [VP 2.143](https://www.wisdomlib.org/hinduism/book/vakyapadiya-of-bhartrihari/d/doc1239250.html), [2.314](https://www.wisdomlib.org/hinduism/book/vakyapadiya-of-bhartrihari/d/doc1239865.html), [2.315–316](https://www.wisdomlib.org/hinduism/book/vakyapadiya-of-bhartrihari/d/doc1239866.html), [3.7.133](https://www.wisdomlib.org/hinduism/book/vakyapadiya-of-bhartrihari/d/doc1337216.html), [Ogawa on Kaiyaṭa and Puṇyarāja](https://www.academia.edu/90739542/Approaching_the_sentence_meaning_in_the_Mah%C4%81bh%C4%81%E1%B9%A3ya_the_V%E1%B9%9Btti_and_the_%E1%B9%AC%C4%ABk%C4%81). Modern competitor: [Causal-JEPA](https://proceedings.mlr.press/v306/nam26c.html), which masks object latents to learn interactions; [WARC](https://arxiv.org/abs/2611.12324).

## Predictions before the run
See frozen protocol. Enumeration: 14 candidate C words and 12 D words, 96,768 joint hypotheses at four slots and 6,048 at three. Source pretraining: 96 worlds per direction; infer word pair frequencies using complete action effects, fractional credit for ambiguity. Target inference: predict D before observing it; subsequently update both D and C posterior using D observations. Forecast +5 pp joint vs uniform, +2 pp joint vs factorized, +2 pp backward vs frozen; unrelated harm at most 2 pp. No GPU scaling justified by this toy alone.

Run the script from this directory; `python -m cProfile -s cumulative e16.py` profiles the full evaluation. For every metric, use the same hidden world and observation stream.

In [ ]:
%run e16.py

## Captured standalone CPU results (not notebook execution counts)
480 test worlds, 192 source worlds. Wall 4.59 s; profiled 6.176 s including imports. All numbers are exact held-out binary scene accuracy deltas in pp.

| Direction / family | Joint–uniform D0 [95% CI] | Joint–factorized D0 [95% CI] | Robust–frozen C2 [95% CI] | Robust–independent D2 [95% CI] |
|---|---:|---:|---:|---:|
| 3→4 matched | 4.297 [1.562,7.187] | 2.344 [.469,4.375] | 8.667 [3,14.833] | 4.494 [-1.336,10.316] |
| 4→3 matched | 11.875 [7.496,16.406] | 5 [1.875,8.438] | 10.357 [3.571,16.429] | 9.881 [4.524,14.704] |
| 3→4 shifted | 2.891 [.857,5.312] | -.156 [-1.719,1.406] | 5.833 [-.833,12.667] | 10.101 [4.422,15.708] |
| 4→3 shifted | 9.375 [4.059,14.844] | 2.188 [-.938,5.312] | 1.071 [-6.071,7.857] | 3.452 [-2.68,9.702] |
| 3→4 independent | .391 [-2.188,2.814] | .312 [-2.031,2.344] | 1.333 [-5.833,8.167] | 1.839 [-3.913,7.376] |
| 4→3 independent | 2.656 [-2.344,8.125] | .625 [-3.125,4.375] | -4.643 [-11.071,2.5] | .774 [-5.119,6.726] |

Gate PASSES by the frozen criteria (null harm point estimate on D2; intervals do NOT establish equivalence). Independent backward C can degrade. Zero D observations: matched joint 40.31% and 65.94%, uniform 36.02% and 54.06%, factorized 37.97% and 60.94%. Two D observations: matched joint 67.98% and 86.07%; independent 63.48% and 76.19%. At four D exposures the gains shrink. Profile: 5,760 posterior calls 2.77s cumulative; 23,040 scoring calls 2.92s. No GPU estimate from these timings.

Limitations: two arities of the same permutation generator, not two independent domains; deterministic worlds, pre-specified word grammar, structured training curriculum, same action positions across worlds; source data identify permutation effects exactly, test data have ambiguity. Independent permutations may coincidentally fit structured hypotheses. Baseline with identical joint prior is identical to candidate. This result cannot demonstrate SOTA or superiority to Causal-JEPA, DreamCoder or WARC. The novel mechanism remains open: discover which whole events become new operands, rather than supplying C as a possible operand in D. Next: learn event-relative participants and discover reusable schemas from full observed transitions; include counterfactual object masking and a conventional joint-inference baseline. No GPU until matched real-data win.